# Hyperliquid Sentiment, Behavior, and Profitability Analysis

This notebook explores how Fear/Greed sentiment affects trader behavior and profitability on Hyperliquid, including EDA, feature engineering, statistical tests, and predictive modeling.


In [ ]:
# Imports and configuration
import os
import pandas as pd
from src.data_prep import prepare_datasets
from src.features import rolling_features, trader_performance_features, build_feature_matrix
from src.eda import descriptive_stats, pnl_by_sentiment, direction_frequency, plot_pnl_distribution, plot_time_series_agg, correlation_heatmap, pairplot_sample
from src.stats_tests import ttest_pnl_fear_vs_greed, correlation_tests, ols_sentiment_on_pnl
from src.modeling import train_regressors, train_classifiers

ROOT = os.path.dirname(os.path.dirname(os.getcwd()))
FG_PATH = os.path.join(ROOT, 'fear_greed_index.csv')
TRADES_PATH = os.path.join(ROOT, 'historical_data.csv')
OUTPUTS = os.path.join(ROOT, 'outputs')
REPORTS = os.path.join(ROOT, 'reports')
os.makedirs(OUTPUTS, exist_ok=True)
os.makedirs(REPORTS, exist_ok=True)


In [ ]:
# Data ingestion & preparation
sentiment_daily, trades_raw, trades = prepare_datasets(FG_PATH, TRADES_PATH)
trades = rolling_features(trades)
trades = trader_performance_features(trades)

sentiment_daily.head(), trades.head()


In [ ]:
# EDA: Descriptive statistics and plots
_ = descriptive_stats(trades)
_.style.background_gradient(cmap='Blues')


In [ ]:
pnl_sent = pnl_by_sentiment(trades)
pnl_sent


In [ ]:
plot_pnl_distribution(trades, out_png=os.path.join(OUTPUTS, 'pnl_distribution.png'))
fig_ts = plot_time_series_agg(trades, out_html=os.path.join(OUTPUTS, 'daily_timeseries.html'))
fig_ts.show()
_ = correlation_heatmap(trades, out_png=os.path.join(OUTPUTS, 'correlation_heatmap.png'))


In [ ]:
# Statistical tests
results_t = ttest_pnl_fear_vs_greed(trades)
results_corr = correlation_tests(trades)
ols_model = ols_sentiment_on_pnl(trades)
results_t, results_corr.head(), ols_model.summary().tables[0]


In [ ]:
# Modeling
X, (y_reg, y_clf), feature_cols = build_feature_matrix(trades)
reg_metrics = train_regressors(X, y_reg, out_dir=os.path.join(OUTPUTS, 'regression'))
clf_metrics = train_classifiers(X, y_clf, out_dir=os.path.join(OUTPUTS, 'classification'))
reg_metrics, clf_metrics


## Executive-Ready Insights

- Summarize how sentiment shifts (Fear → Greed) align with trader profitability, win rates, and volume.
- Highlight which features most influence profitability (from SHAP/model coefficients).
- Recommend sentiment-aware risk controls (e.g., reduce leverage during extreme greed).
